In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from matplotlib.animation import FuncAnimation, PillowWriter
from google.colab import files

# 1) Upload CSV
uploaded = files.upload()  # ↳ select your comparison.csv
csv_path = next(iter(uploaded))
df = pd.read_csv(csv_path)

# 2) Precompute axis limits
x_all = df[['true_x','ppo_x','ridl_x']].values.flatten()
y_all = df[['true_y','ppo_y','ridl_y']].values.flatten()
z_all = df[['true_z','ppo_z','ridl_z']].values.flatten()
x_min, x_max = x_all.min(), x_all.max()
y_min, y_max = y_all.min(), y_all.max()
z_min, z_max = z_all.min(), z_all.max()

# 3) Set up figure and axes
fig = plt.figure(figsize=(10, 5), facecolor='black')
ax3d = fig.add_subplot(1, 2, 1, projection='3d', facecolor='black')
ax2d = fig.add_subplot(1, 2, 2, facecolor='black')
for ax in (ax3d, ax2d):
    ax.tick_params(colors='white')
    ax.xaxis.label.set_color('white')
    ax.yaxis.label.set_color('white')

ax3d.set_xlim(x_min, x_max)
ax3d.set_ylim(y_min, y_max)
ax3d.set_zlim(z_min, z_max)
ax3d.set_xlabel('X'); ax3d.set_ylabel('Y'); ax3d.set_zlabel('Z')
ax2d.set_xlabel('Step'); ax2d.set_ylabel('Error')

# 4) Initialize lines
line_true, = ax3d.plot([], [], [], color='white', lw=2, label='TRUE')
line_ppo,  = ax3d.plot([], [], [], color='red',   lw=2, label='PPO')
line_ridl, = ax3d.plot([], [], [], color='blue',  lw=2, label='RIDL')
err_true_line, = ax2d.plot([], [], color='white', lw=2, label='ERR_TRUE')
err_ridl_line, = ax2d.plot([], [], color='blue',  lw=2, label='ERR_RIDL')

ax3d.legend(facecolor='black', labelcolor='white')
ax2d.legend(facecolor='black', labelcolor='white')

# 5) Storage for animated data
t_true, t_ppo, t_ridl = ([], [], []), ([], [], []), ([], [], [])
err_true_list, err_ridl_list = [], []

def init():
    line_true.set_data([], []); line_true.set_3d_properties([])
    line_ppo.set_data([], []);  line_ppo.set_3d_properties([])
    line_ridl.set_data([], []); line_ridl.set_3d_properties([])
    err_true_line.set_data([], [])
    err_ridl_line.set_data([], [])
    return line_true, line_ppo, line_ridl, err_true_line, err_ridl_line

def update(i):
    # Append trajectory
    t_true[0].append(df.loc[i,'true_x']); t_true[1].append(df.loc[i,'true_y']); t_true[2].append(df.loc[i,'true_z'])
    t_ppo[0].append(df.loc[i,'ppo_x']);   t_ppo[1].append(df.loc[i,'ppo_y']);   t_ppo[2].append(df.loc[i,'ppo_z'])
    t_ridl[0].append(df.loc[i,'ridl_x']); t_ridl[1].append(df.loc[i,'ridl_y']); t_ridl[2].append(df.loc[i,'ridl_z'])
    err_true_list.append(df.loc[i,'err_true'])
    err_ridl_list.append(df.loc[i,'err_ridl'])

    line_true.set_data(t_true[0], t_true[1]); line_true.set_3d_properties(t_true[2])
    line_ppo.set_data(t_ppo[0],   t_ppo[1]);   line_ppo.set_3d_properties(t_ppo[2])
    line_ridl.set_data(t_ridl[0], t_ridl[1]); line_ridl.set_3d_properties(t_ridl[2])

    steps = list(range(len(err_true_list)))
    err_true_line.set_data(steps, err_true_list)
    err_ridl_line.set_data(steps, err_ridl_list)
    ax2d.set_xlim(0, len(err_true_list))
    ax2d.set_ylim(min(err_true_list+err_ridl_list), max(err_true_list+err_ridl_list))
    return line_true, line_ppo, line_ridl, err_true_line, err_ridl_line

# 6) Create animation and save as GIF
anim = FuncAnimation(fig, update, frames=len(df), init_func=init, blit=True)
gif_path = '/content/trajectory_animation.gif'
anim.save(gif_path, writer=PillowWriter(fps=10))

print(f"Saved animation to {gif_path}")
print('NVR on VR')